In [ ]:
from google.colab import drive
drive.mount('/content/drive')



In [ ]:
file_path = '/content/drive/My Drive/Python人工智能PyTorch教程素材/第3章/dataloader处理方法.py'

with open(file_path, 'r', encoding='utf-8') as file:
    content = file.read()

# 显示文件内容
print(content)


In [ ]:
#任务：是
# import os
#
# def train_test_file(root,dir):
#     file_txt = open(dir+'.txt','w')
#     path = os.path.join(root,dir)
#     for roots,directories, files in os.walk(path):
#         if len(directories) != 0 :
#             dirs = directories
#         else:
#             now_dir = roots.split('\\')
#             for file in files:
#                 path_1 = os.path.join(roots,file)
#                 print(path_1)
#                 file_txt.write(path_1+' '+str(dirs.index(now_dir[-1]))+'\n')
#     file_txt.close()
# root = r'.\食物分类\food_dataset2'
# train_dir = 'train'
# test_dir = 'test'
# train_test_file(root,train_dir)
# train_test_file(root,test_dir)





'''创建数据集的类'''
import torch
from torch.utils.data import Dataset,DataLoader     #用于处理数据集的
import numpy as np
from PIL import Image
from torchvision import transforms      #对数据进行处理工具  转换


data_transforms = {     #字典
    'train':
        transforms.Compose([        #组合，
        transforms.Resize([256,256]),   #数据进行改变大小[256,256]，
        transforms.ToTensor(),          #数据转换为tensor，默认把通道维度放在前面
    ]),
    'valid':
        transforms.Compose([
        transforms.Resize([256,256]),
        transforms.ToTensor(),
    ]),
}

class food_dataset(Dataset):    #food_dataset是自己创建的类名称，可以改为你需要的名称
    def __init__(self, file_path,transform=None): #类的初始化
        self.file_path = file_path
        self.imgs = []
        self.labels = []
        self.transform = transform
        with open(self.file_path) as f:
            samples = [x.strip().split(' ') for x in f.readlines()]
            for img_path, label in samples:
                self.imgs.append(img_path)  #图像的路径
                self.labels.append(label)   #标签,还不是tensor

    def __len__(self):  #类实例化对象后，可以使用len函数测量对象的个数
        return len(self.imgs)

    def __getitem__(self, idx): #关键，可通过索引的形式获取每一个图片数据及标签
        image = Image.open(self.imgs[idx])   #读取到图片数据，还不是tensor
        if self.transform:                   #将pil图像数据转换为tensor
            image = self.transform(image)    #图像处理为256*256，转换为tenor

        label = self.labels[idx]        #label还不是tensor
        label = torch.from_numpy(np.array(label,dtype = np.int64))  #label也转换为tensor
        return image, label



training_data = food_dataset(file_path = './train.txt',transform = data_transforms['train'])
test_data = food_dataset(file_path = './test.txt',transform = data_transforms['valid'])

#training_data需要具备索引的功能，还要确保数据是tensor
train_dataloader = DataLoader(training_data, batch_size=64,shuffle=True)#64张图片为一个包，
test_dataloader = DataLoader(test_data, batch_size=64,shuffle=True)


'''展示手写字图片，把训练数据集中的前9张图片展示一下'''
# from matplotlib import pyplot as plt
# image, label = iter(train_dataloader).__next__()        #iter是一个迭代器函数。__next__()用于获取下一个数据
# sample = image[2]
# sample = sample.permute((1, 2, 0)).numpy()  #tensor数据的维度转换
# plt.imshow(sample)
# plt.show()
# print('Label is: {}'.format(label[2].numpy()))

'''-------------cnn卷积神经网络部分----------------------'''

'''判断当前设备是否支持GPU，其中mps是苹果m系列芯片的GPU。'''
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using {device} device")

''' 定义神经网络  '''

from torch import nn

class CNN(nn.Module):
    def __init__(self):         # 输入大小 (3, 256, 256)
        super(CNN, self).__init__()
        self.conv1 = nn.Sequential(  #将多个层组合成一起。
            nn.Conv2d(          #2d一般用于图像，3d用于视频数据（多一个时间维度），1d一般用于结构化的序列数据
                in_channels=3,  # 图像通道个数，1表示灰度图（确定了卷积核 组中的个数），
                out_channels=16,# 要得到几多少个特征图，卷积核的个数
                kernel_size=5,  # 卷积核大小，5*5
                stride=1,       # 步长
                padding=2,      # 一般希望卷积核处理后的结果大小与处理前的数据大小相同，效果会比较好。那padding改如何设计呢？建议stride为1，kernel_size = 2*padding+1
            ),                  # 输出的特征图为 (16, 256, 256)
            nn.ReLU(),  # relu层
            nn.MaxPool2d(kernel_size=2),  # 进行池化操作（2x2 区域）, 输出结果为： (16, 128, 128)
        )
        self.conv2 = nn.Sequential(  #输入 (16, 128, 128)
            nn.Conv2d(16, 32, 5, 1, 2),  # 输出 (32, 128, 128)
            nn.ReLU(),  # relu层
            nn.Conv2d(32, 32, 5, 1, 2), # 输出 (32, 128, 128)
            nn.ReLU(),
            nn.MaxPool2d(2),  # 输出 (32, 64, 64)
        )

        self.conv3 = nn.Sequential(  #输入 (32, 64, 64)
            nn.Conv2d(32, 64, 5, 1, 2),
            nn.ReLU(),  # 输出 (64, 64, 64)
        )

        self.out = nn.Linear(64 * 64 * 64, 20)  # 全连接层得到的结果

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)# 输出 (64,64, 32, 32)
        x = x.view(x.size(0), -1)  # flatten操作，结果为：(batch_size, 64 * 32 * 32)
        output = self.out(x)
        return output

model = CNN().to(device)
print(model)

def train(dataloader, model, loss_fn, optimizer):
    model.train()
#pytorch提供2种方式来切换训练和测试的模式，分别是：model.train() 和 model.eval()。
# 一般用法是：在训练开始之前写上model.trian()，在测试时写上 model.eval() 。
    batch_size_num = 1
    for X, y in dataloader:                 #其中batch为每一个数据的编号
        X, y = X.to(device), y.to(device)   #把训练数据集和标签传入cpu或GPU
        pred = model.forward(X)             #自动初始化 w权值
        loss = loss_fn(pred, y)             #通过交叉熵损失函数计算损失值loss
        # Backpropagation 进来一个batch的数据，计算一次梯度，更新一次网络
        optimizer.zero_grad()               #梯度值清零
        loss.backward()                     #反向传播计算得到每个参数的梯度值
        optimizer.step()                    #根据梯度更新网络参数

        loss = loss.item()                  #获取损失值
        if batch_size_num %100 == 0:
            print(f"loss: {loss:>7f}  [number:{batch_size_num}]")
        batch_size_num += 1

def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()    #
    test_loss, correct = 0, 0
    with torch.no_grad():   #一个上下文管理器，关闭梯度计算。当你确认不会调用Tensor.backward()的时候。这可以减少计算所用内存消耗。
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model.forward(X)
            test_loss += loss_fn(pred, y).item() #
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
            a = (pred.argmax(1) == y)  #dim=1表示每一行中的最大值对应的索引号，dim=0表示每一列中的最大值对应的索引号
            b = (pred.argmax(1) == y).type(torch.float)
    test_loss /= num_batches
    correct /= size
    print(f"Test result: \n Accuracy: {(100*correct)}%, Avg loss: {test_loss}")

loss_fn = nn.CrossEntropyLoss() #创建交叉熵损失函数对象，因为手写字识别中一共有10个数字，输出会有10个结果
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)#创建一个优化器，SGD为随机梯度下降算法？？
#params：要训练的参数，一般我们传入的都是model.parameters()。
#lr：learning_rate学习率，也就是步长。


# train(train_dataloader, model, loss_fn, optimizer)
# test(test_dataloader, model, loss_fn)


epochs = 50
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    # test(test_dataloader, model, loss_fn)
print("Done!")
test(test_dataloader, model, loss_fn)

'''
总结：
    1、一定要把train.txt和test.txt文件实现出来
    2、你需要创建Dataset数据，你自己继承。过程中需要保证数据为tensor
    3、排查一下数据是否正确，3，256，256，
    4、神经网络里面的参数需要仔细的确定。
'''

'''补充1个类中__getitem__小知识'''
# class USE_getitem():
#     def __init__(self, text):
#         self.text = text
#     def __getitem__(self, index):     #可以通过列表索引的方式获取类对象的数据
#         result = self.text[0].upper()#字符串中一个方法，将字符串大写
#         return result
#
# p = USE_getitem("pytorch")  #对象数据，
# print(p[0],p[1])    #字符串，列表


'''补充1个PIL（pillow图像处理库）的使用介绍
PIL可以做很多和图像处理相关的事情:
    1、图像归档(Image Archives)：PIL非常适合于图像归档以及图像的批处理任务。你可以使用PIL创建缩略图，转换图像格式，打印图像等等。
    2、图像展示(Image Display)：支持包括Tk PhotoImage，BitmapImage还有Windows DIB等接口。PIL支持众多的GUI框架接口，可以用于图像展示。
    3、图像处理(Image Processing)：支持图像的大小转换，图像旋转，以及任意的仿射变换。PIL还有一些直方图的方法，允许你展
    4、示图像的一些统计特性。这个可以用来实现图像的自动对比度增强，还有全局的统计分析等。
'''
# from PIL import Image
# img = Image.open(r'.\食物分类\food_dataset\train\薯条\img_薯条_35.jpeg')
# img.show()#图片显示
# print(img.format, img.size, img.mode)   #打印图片属性
# img.save(r'.\食物分类\food_dataset\train\薯条\img_薯条_36.png') #保存为






















